In [1]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

Project root: C:\Users\markd\OneDrive\Desktop\Claude\GitHub\Projects\uk-energy-price-forecasting


# 05 — Probabilistic Evaluation

Point forecasts are cheap to evaluate (MAE, RMSE).  This notebook evaluates **probabilistic calibration**:

- **Reliability diagram**: nominal vs empirical coverage at multiple   interval levels — a well-calibrated model's dots should fall on the   diagonal.
- **Fan chart**: median + quantile bands for one origin's 48-step   forecast vs actual prices.

### Calibration note
> `SeasonalNaive` produces *zero-width* intervals: all quantiles equal the same value, so empirical coverage is ~0 for any nominal level.  This is itself a teaching point — a model can have excellent MAE yet completely fail calibration.  `GlobalLGBM` with independent quantile regressors should achieve reasonable (if not perfect) calibration on the fixture panel.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.build.fixtures import load_fixture_panel
from src.models.baselines import SeasonalNaive
from src.models.global_ml import GlobalLGBM
from src.backtest.rolling_origin import generate_origins, run_backtest
from src.metrics.coverage import interval_coverage

import warnings
warnings.filterwarnings('ignore')

bundle = load_fixture_panel()
target_idx = bundle.target.time_index
print('Fixture bundle loaded.')

The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.


The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.


The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.


Fixture bundle loaded.


## Run backtest (re-run if coming from fresh kernel)

In [3]:
HORIZON   = 48
QUANTILES = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)

origins = generate_origins(
    time_index=target_idx,
    # Darts strips tz from TimeSeries.time_index; pass tz-naive timestamps.
    start=pd.Timestamp('2024-01-20 00:00'),
    end=pd.Timestamp('2024-01-25 00:00'),
    step='1D',
)
print(f'{len(origins)} origins for probabilistic evaluation.')

result_naive = run_backtest(
    model=SeasonalNaive(season=48),
    bundle=bundle,
    origins=origins,
    horizon=HORIZON,
    quantiles=QUANTILES,
    refit=False,
    model_name='seasonal_naive',
)

lgbm = GlobalLGBM(
    quantiles=list(QUANTILES),
    lags=[-1, -2, -3, -48, -96],
    lags_past_covariates=[-1, -2, -3, -48],
    n_estimators=50,
    num_leaves=15,
    verbose=-1,
)
result_lgbm = run_backtest(
    model=lgbm,
    bundle=bundle,
    origins=origins,
    horizon=HORIZON,
    quantiles=QUANTILES,
    refit=False,
    model_name='global_lgbm',
)
print('Backtests complete.')

6 origins for probabilistic evaluation.


Backtests complete.


## Reliability diagram

For each symmetric nominal interval `[q_low, q_high]` (e.g. 80% = [0.1, 0.9]) we compute the empirical fraction of actuals falling inside the interval.  A perfectly calibrated model lies on the diagonal.

In [4]:
def reliability_points(result, quantile_pairs):
    """Return (nominal_coverage, empirical_coverage) for each quantile pair."""
    actuals = result.actuals.flatten()
    fc      = result.forecasts
    nominals, empiricals = [], []
    for (q_lo, q_hi) in quantile_pairs:
        lower = fc[q_lo].flatten()
        upper = fc[q_hi].flatten()
        emp   = interval_coverage(actuals, lower, upper)
        nominals.append(q_hi - q_lo)
        empiricals.append(emp)
    return np.array(nominals), np.array(empiricals)

# Symmetric pairs centred on 0.5
pairs = [
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.1, 0.9),
]

nom_naive, emp_naive = reliability_points(result_naive, pairs)
nom_lgbm,  emp_lgbm  = reliability_points(result_lgbm,  pairs)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.plot(nom_naive, emp_naive, 'o-', color='grey',      lw=1.5,
        markersize=7, label='SeasonalNaive (zero-width intervals)')
ax.plot(nom_lgbm,  emp_lgbm,  's-', color='royalblue', lw=1.5,
        markersize=7, label='GlobalLGBM')

ax.set_xlabel('Nominal coverage')
ax.set_ylabel('Empirical coverage')
ax.set_title('Reliability diagram — probabilistic calibration')
ax.legend(fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
fig.tight_layout()

Path('reports/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('reports/figures/05_reliability_diagram.png', dpi=120)
plt.show()
print('Saved: reports/figures/05_reliability_diagram.png')

# Print a quick table
cov_df = pd.DataFrame({
    'nominal'   : nom_lgbm,
    'naive_emp' : emp_naive,
    'lgbm_emp'  : emp_lgbm,
})
print(cov_df.round(3).to_string(index=False))

Saved: reports/figures/05_reliability_diagram.png
 nominal  naive_emp  lgbm_emp
     0.2        0.0     0.198
     0.4        0.0     0.347
     0.6        0.0     0.590
     0.8        0.0     0.747


## Fan chart — one origin

A fan chart shows the median forecast + shaded interval bands for a single forecast origin.  This lets us visually inspect how the uncertainty widens across the 48-step horizon.

In [5]:
# Pick the first valid origin from the LGBM backtest
origin_idx = 0
origin = result_lgbm.origins[origin_idx]

# Actual prices for the horizon window
actuals_window = result_lgbm.actuals[origin_idx]       # shape (48,)
sp_axis = np.arange(1, HORIZON + 1)

# Quantile arrays
q10 = result_lgbm.forecasts[0.1][origin_idx]
q20 = result_lgbm.forecasts[0.2][origin_idx]
q30 = result_lgbm.forecasts[0.3][origin_idx]
q40 = result_lgbm.forecasts[0.4][origin_idx]
q50 = result_lgbm.forecasts[0.5][origin_idx]
q60 = result_lgbm.forecasts[0.6][origin_idx]
q70 = result_lgbm.forecasts[0.7][origin_idx]
q80 = result_lgbm.forecasts[0.8][origin_idx]
q90 = result_lgbm.forecasts[0.9][origin_idx]

fig, ax = plt.subplots(figsize=(12, 4))

# Shaded bands (darkest = tightest)
alpha_levels = [0.15, 0.20, 0.25, 0.30]
band_pairs   = [(q10, q90), (q20, q80), (q30, q70), (q40, q60)]
labels       = ['10–90%', '20–80%', '30–70%', '40–60%']
for (lo, hi), a, lbl in zip(band_pairs, alpha_levels, labels):
    ax.fill_between(sp_axis, lo, hi, alpha=a, color='royalblue', label=lbl)

ax.plot(sp_axis, q50, color='royalblue', lw=2, label='Median (q50)')
ax.plot(sp_axis, actuals_window, color='black', lw=1.5, ls='--', label='Actual')

ax.set_xlabel('Forecast step (SP offset from origin)')
ax.set_ylabel('Price (£/MWh)')
ax.set_title(f'Fan chart — GlobalLGBM, origin {origin}')
ax.legend(ncol=3, fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('reports/figures/05_fan_chart.png', dpi=120)
plt.show()
print('Saved: reports/figures/05_fan_chart.png')

Saved: reports/figures/05_fan_chart.png


## SeasonalNaive calibration teaching point

Let's confirm that seasonal-naive has zero-width intervals and therefore ~0 empirical coverage at all levels.

In [6]:
# All quantiles for SeasonalNaive should be identical (zero interval width)
o = 0   # first origin
q10_n = result_naive.forecasts[0.1][o]
q90_n = result_naive.forecasts[0.9][o]
width = q90_n - q10_n
print(f'SeasonalNaive 80% interval width (first origin): '
      f'min={width.min():.4f}  max={width.max():.4f}')
print('Confirms: zero-width intervals => empirical coverage ≈ 0')

print(f'\nGlobalLGBM 80% empirical coverage (all origins): '
      f'{emp_lgbm[-1]:.3f} (nominal 0.80)')

SeasonalNaive 80% interval width (first origin): min=0.0000  max=0.0000
Confirms: zero-width intervals => empirical coverage ≈ 0

GlobalLGBM 80% empirical coverage (all origins): 0.747 (nominal 0.80)
